# This notebook generates CMG dat files for running CMG simulations

## Step 1: Set up a base CMG model
* Prepare a base CMG dat file and add it to data/dat_file_templates
* Note CMG requires initializing stress state using a reference block. For the JD_Sula_2005_gmc grid, there are 10 k layers. Reservoir starts at k=6. Use block (50, 1, 6) as reference block for *STRESSGRAD calculation. Its grid top = 670.7188 m and bottom = 671.9521 m.

## Step 2: Sample uncertain parameters

* Note PORO/PERMX pairs are NOT sampled more than once (from file names instead of files)

### Option 1: Monte Carlo sampling

In [1]:
from pathlib import Path
import sys

base_path = Path('..')
repo_root = Path.cwd().parent
sys.path.append(str(repo_root / "src"))

from parameter_sampling import latin_hypercube_sampling

# Note: 1) stress gradients are effective ones (required by CMG) after subtracting 10; 
#       2) stress gradients are negative due to CMG DIR DOWN convention
# OMV_values = [20e6, 0.3, x, 14.6, 22.7, 300]
# base_values = [20e6, 0.3, 28, 16.5, 22.7, 310]
sampling_results = latin_hypercube_sampling(
    name_prefix = 'test_250922', # prefix for the output file name
    random_seed = 11, # random seed for Latin Hypercube sampling
    property_file_names_path = base_path/'results'/'property_file_names'/'property_file_names_seed0&1.csv', # path for the property file names
    output_file_path = base_path/'results'/'sim_files', # path for the output file
    n_samples = 90,  # number of unique poro/permx pairs to sample (should be <= number of available pairs)
    param_names = ['E_GPa', 'PR', 'SH_MPa/km', 'Sh_MPa/km', 'Sv_MPa/km', 'SH_azi_deg'],
    lower_bounds = [15e6, 0.2, -18 * 1.1, -6.5 * 1.1, -12.7 * 1.1, 300],
    upper_bounds = [25e6, 0.4, -18 * 0.9, -6.5 * 0.9, -12.7 * 0.9, 320],
    # param_names = ['E_GPa', 'PR', 'SH_MPa/km', 'Sh_MPa/km', 'Sv_MPa/km', 'SH_azi_deg', 'A_m2']
    # lower_bounds = [15e6, 0.2, -18 * 1.1, -6.5 * 1.1, -12.7 * 1.1, 300, 16985344.51*0.9]
    # upper_bounds = [25e6, 0.4, -18 * 0.9, -6.5 * 0.9, -12.7 * 0.9, 320, 16985344.51*1.1]
    ref_block_top_depth = 670.7188,   # initial stress reference block, for the JD_Sula_2005_gmc grid, the reference block is (50, 1, 6) 
    ref_block_bottom_depth = 671.9521,
    show_results = True
)

,E_GPa,PR,SH_MPa/km,Sh_MPa/km,Sv_MPa/km,SH_azi_deg,PORO_file,PERMX_file,beta,cos_2beta,sin_2beta,sigma_x_grad,sigma_y_grad,tau_xy_grad,sigma_x_ref,sigma_y_ref,sigma_z_ref,tau_xy_ref
0,17430158.87,0.32,-18.98,-6.66,-12.37,306.90,data_properties/JD_BASECASE_5_PORO.dat,data_properties/JD_BASECASE_5_PERMX.dat,216.90,0.28,0.96,-14.54,-11.10,5.92,9761.17,7454.13,8301.40,-3972.95
1,18547731.05,0.23,-17.96,-6.08,-12.80,310.55,data_properties/JD_BASECASE_6_PORO.dat,data_properties/JD_BASECASE_6_PERMX.dat,220.55,0.15,0.99,-12.94,-11.10,5.87,8683.94,7450.65,8589.79,-3940.54
2,21148573.01,0.27,-17.25,-6.54,-12.92,318.11,data_properties/JD_BASECASE_7_PORO.dat,data_properties/JD_BASECASE_7_PERMX.dat,228.11,-0.11,0.99,-11.31,-12.47,5.32,7594.99,8373.25,8671.29,-3572.27
3,15242584.84,0.31,-16.44,-6.13,-11.98,307.23,data_properties/JD_BASECASE_8_PORO.dat,data_properties/JD_BASECASE_8_PERMX.dat,217.23,0.27,0.96,-12.67,-9.90,4.97,8502.71,6647.08,8043.84,-3334.68
4,18738525.02,0.32,-18.81,-6.38,-13.77,301.53,data_properties/JD_BASECASE_9_PORO.dat,data_properties/JD_BASECASE_9_PERMX.dat,211.53,0.45,0.89,-15.41,-9.78,5.54,10346.18,6565.43,9243.42,-3718.72
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,23335499.95,0.31,-18.69,-6.26,-12.77,311.18,data_properties/JD_BASECASE_194_PORO.dat,data_properties/JD_BASECASE_194_PERMX.dat,221.18,0.13,0.99,-13.30,-11.65,6.16,8928.93,7819.85,8575.10,-4133.39
86,24592116.34,0.30,-19.11,-5.92,-12.21,301.78,data_properties/JD_BASECASE_197_PORO.dat,data_properties/JD_BASECASE_197_PERMX.dat,211.78,0.45,0.90,-15.45,-9.58,5.90,10370.73,6430.55,8197.62,-3962.91
87,16276499.15,0.30,-19.69,-5.99,-13.53,312.25,data_properties/JD_BASECASE_198_PORO.dat,data_properties/JD_BASECASE_198_PERMX.dat,222.25,0.10,1.00,-13.50,-12.18,6.81,9060.04,8178.87,9083.68,-4575.02
88,20519907.95,0.39,-18.75,-6.28,-12.94,307.56,data_properties/JD_BASECASE_199_PORO.dat,data_properties/JD_BASECASE_199_PERMX.dat,217.56,0.26,0.97,-14.12,-10.92,6.03,9477.78,7327.78,8689.28,-4047.01


### Option 2: Monte Carlo sampling + importance sampling

In [3]:
from pathlib import Path
import sys

base_path = Path('..')
repo_root = Path.cwd().parent
sys.path.append(str(repo_root / "src"))

from parameter_sampling import latin_hypercube_and_importance_sampling

# Note: 1) stress gradients are effective ones (required by CMG) after subtracting 10; 
#       2) stress gradients are negative due to CMG DIR DOWN convention
# OMV_values = [20e6, 0.3, x, 14.6, 22.7, 300]
# base_values = [20e6, 0.3, 28, 16.5, 22.7, 310]
sampling_results = latin_hypercube_and_importance_sampling(
    name_prefix = 'test_251209', # prefix for the output file name
    random_seed = 15, # random seed for Latin Hypercube sampling
    property_file_names_path = base_path/'results'/'property_file_names'/'property_file_names_seed7.csv', # path for the property file names
    output_file_path = base_path/'results'/'sim_files', # path for the output file
    n_samples = 83,  # number of unique poro/permx pairs to sample (should be <= number of available pairs)
    param_names = ['E_GPa', 'PR', 'SH_MPa/km', 'Sh_MPa/km', 'Sv_MPa/km', 'SH_azi_deg', 'A_m2'],
    lower_bounds = [15e6, 0.2, -18 * 1.1, -6.5 * 1.1, -12.7 * 1.1, 300, 16985344.51*0.9],
    upper_bounds = [25e6, 0.4, -18 * 0.9, -6.5 * 0.9, -12.7 * 0.9, 320, 16985344.51*1.1],
    ref_block_top_depth = 670.7188,   # initial stress reference block, for the JD_Sula_2005_gmc grid, the reference block is (50, 1, 6) 
    ref_block_bottom_depth = 671.9521,
    show_results = True,
    # below is for importance sampling for the Sula CCS
    alpha = 0.9,
    beta = 0.9,
    proposal_SH_azi_low = 319,
    proposal_SH_low = 18*1.05,
    show_summary = True
)

SH_azi
Target distribution: U[300, 320], 83 samples
Proposal distribution: 0.1 * U[300, 319] + 0.9 * U[319])
Importance samples min: 300.03, max: 319.99, number of alpha samples: 74
SH
Target distribution: U[16.2, 19.8], 83 samples
Proposal distribution: 0.1 * U[16.2, 18.90] + 0.9 * U[18.90, 19.8]
Importance samples min: 16.34, max: 19.80, number of beta samples: 74


,E_GPa,PR,SH_MPa/km,Sh_MPa/km,Sv_MPa/km,SH_azi_deg,A_m2,PORO_file,PERMX_file,beta,cos_2beta,sin_2beta,sigma_x_grad,sigma_y_grad,tau_xy_grad,sigma_x_ref,sigma_y_ref,sigma_z_ref,tau_xy_ref
0,23952681.52,0.27,-19.62,-5.87,-12.06,319.42,16035034.07,data_properties/JD_BASECASE_723_PORO.dat,data_properties/JD_BASECASE_723_PERMX.dat,229.42,-0.15,0.99,-11.69,-13.80,6.80,7845.43,9264.54,8096.00,-4561.83
1,19295740.18,0.20,-19.26,-6.27,-12.13,319.50,18317911.44,data_properties/JD_BASECASE_724_PORO.dat,data_properties/JD_BASECASE_724_PERMX.dat,229.50,-0.16,0.99,-11.75,-13.78,6.42,7888.10,9252.28,8142.68,-4307.61
2,17045984.84,0.35,-18.95,-6.35,-12.49,319.57,16746690.26,data_properties/JD_BASECASE_725_PORO.dat,data_properties/JD_BASECASE_725_PERMX.dat,229.57,-0.16,0.99,-11.65,-13.66,6.22,7823.13,9167.86,8382.86,-4175.48
3,22119443.23,0.24,-19.44,-6.69,-11.64,319.88,17513822.51,data_properties/JD_BASECASE_726_PORO.dat,data_properties/JD_BASECASE_726_PERMX.dat,229.88,-0.17,0.99,-11.99,-14.15,6.28,8046.86,9498.01,7812.72,-4217.85
4,19909813.35,0.38,-19.06,-6.77,-11.46,319.20,17032079.97,data_properties/JD_BASECASE_727_PORO.dat,data_properties/JD_BASECASE_727_PERMX.dat,229.20,-0.15,0.99,-12.02,-13.81,6.08,8066.27,9271.55,7695.46,-4080.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,19518194.14,0.21,-17.96,-6.40,-13.29,300.03,16197989.87,data_properties/JD_BASECASE_818_PORO.dat,data_properties/JD_BASECASE_818_PERMX.dat,210.03,0.50,0.87,-15.06,-9.30,5.01,10111.56,6241.44,8920.30,-3360.10
79,18464827.66,0.38,-19.29,-6.93,-11.91,301.36,15998887.30,data_properties/JD_BASECASE_819_PORO.dat,data_properties/JD_BASECASE_819_PERMX.dat,211.36,0.46,0.89,-15.95,-10.28,5.49,10704.85,6901.67,7997.91,-3688.34
80,17758646.58,0.26,-16.66,-6.82,-13.31,319.08,17074167.83,data_properties/JD_BASECASE_820_PORO.dat,data_properties/JD_BASECASE_820_PERMX.dat,229.08,-0.14,0.99,-11.04,-12.44,4.87,7412.69,8350.58,8934.62,-3271.25
81,22621006.12,0.28,-19.70,-6.34,-11.84,319.18,16895808.88,data_properties/JD_BASECASE_821_PORO.dat,data_properties/JD_BASECASE_821_PERMX.dat,229.18,-0.15,0.99,-12.04,-13.99,6.61,8085.12,9390.65,7948.96,-4436.91


## Step 3: generate CMG dat files based on the sampled parameters

In [ ]:
import pandas as pd
from pathlib import Path
import sys

repo_root = Path.cwd().parent
sys.path.append(str(repo_root / "src"))
base_path = Path('..')

from generate_dat_files import generate_dat_files

# df_params = pd.read_csv(base_path/'results'/'sim_files'/f"{sampling_results['name_prefix']}_sampled_params_seed{sampling_results['random_seed']}.csv")
generate_dat_files(
    df_parameters = sampling_results['param_dataframe'],
    template_file_path = base_path/'data'/'dat_file_templates'/'250913.dat',
    save_folder_path = base_path/'results'/'sim_files'/f"{sampling_results['name_prefix']}_dat_files"
)

Generated 83 dat files successfully.
